# Xeno-canto Gathering

Build a reproducible download set using XC query tags.


In [1]:
import json
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from src.config import CONFIG
from src.dataset.utils.xeno_canto import write_raw_manifest_for_species, manifest_path
from src.dataset.utils.selection import analyze_manifests_dir, aggregate_overall_stats


In [2]:
DATA_DIR = Path(CONFIG.paths.data_dir)
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir

for p in [RAW_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    
SPECIES_FILE = Path("../../species_list_small.json")
species_map = json.loads(SPECIES_FILE.read_text(encoding="utf-8"))
species_list = list(species_map.values())
species_list[:5]

[{'common_name': 'European Robin', 'sci_name': 'Erithacus rubecula'},
 {'common_name': 'Eurasian Blackbird', 'sci_name': 'Turdus merula'},
 {'common_name': 'Eurasian Wren', 'sci_name': 'Troglodytes troglodytes'},
 {'common_name': 'Eurasian Blue Tit', 'sci_name': 'Cyanistes caeruleus'},
 {'common_name': 'Great Tit', 'sci_name': 'Parus major'}]

In [3]:
def build_query(sci_name: str) -> str:
    xc = CONFIG.xeno_canto
    tags = [
        f'sp:"{sci_name}"',
        "grp:birds",
        "area:europe",
        'q:">C"',                 # keep this if you want
        f'len:">{xc.min_len}"',
        f'len:"<{xc.max_len}"',
    ]
    return " ".join(tags)


## Fetch metadata and store in manifest files


In [4]:
for entry in species_list:
    sci_name = entry["sci_name"]
    query = build_query(sci_name)

    out_csv = write_raw_manifest_for_species(
        sci_name=sci_name,
        query=query,
        manifest_dir=MANIFEST_DIR,
        per_page=500,
    )

    print(f"Wrote raw manifest: {out_csv}")


Wrote raw manifest: bird_data\manifests\erithacus_rubecula.csv
Wrote raw manifest: bird_data\manifests\turdus_merula.csv
Wrote raw manifest: bird_data\manifests\troglodytes_troglodytes.csv
Wrote raw manifest: bird_data\manifests\cyanistes_caeruleus.csv
Wrote raw manifest: bird_data\manifests\parus_major.csv
Wrote raw manifest: bird_data\manifests\carduelis_carduelis.csv
Wrote raw manifest: bird_data\manifests\fringilla_coelebs.csv
Wrote raw manifest: bird_data\manifests\turdus_philomelos.csv
Wrote raw manifest: bird_data\manifests\pica_pica.csv
Wrote raw manifest: bird_data\manifests\coloeus_monedula.csv
Wrote raw manifest: bird_data\manifests\columba_livia.csv
Wrote raw manifest: bird_data\manifests\columba_palumbus.csv
Wrote raw manifest: bird_data\manifests\streptopelia_decaocto.csv
Wrote raw manifest: bird_data\manifests\sturnus_vulgaris.csv
Wrote raw manifest: bird_data\manifests\passer_domesticus.csv
Wrote raw manifest: bird_data\manifests\phylloscopus_trochilus.csv
Wrote raw man

In [5]:
stats = analyze_manifests_dir(MANIFEST_DIR, top_n=10)
print(stats["overall"])

# Save to inspect
(Path("bird_data") / "manifest_stats.json").write_text(
    json.dumps(stats, indent=2),
    encoding="utf-8"
)

{'files': 50, 'total_rows': 46218, 'single_rows': 32683, 'multi_rows': 13535, 'single_pct': 70.71487299320611, 'multi_pct': 29.285127006793886}


164714

In [6]:
stats2 = aggregate_overall_stats(stats, top_n=15)

print(stats2["overall_aggregates"]["countries"]["top"])
print(stats2["overall_aggregates"]["months"]["counts"])
print("Ireland+UK %:", stats2["overall_aggregates"]["countries"]["ireland_uk_pct"])

(Path("bird_data") / "manifest_stats_aggregated.json").write_text(
    json.dumps(stats2, indent=2),
    encoding="utf-8"
)

[('France', 8709), ('Germany', 6808), ('United Kingdom', 5067), ('Poland', 4354), ('Spain', 4182), ('Sweden', 3170), ('Netherlands', 2812), ('Portugal', 2700), ('Belgium', 2100), ('Italy', 1744), ('Ireland', 1577), ('Finland', 847), ('Norway', 771), ('Switzerland', 614), ('Denmark', 553)]
{'09': 2915, '07': 3226, '08': 2586, '06': 4815, '04': 7062, '10': 3933, '05': 7336, '11': 2387, '03': 5349, '01': 2000, '02': 2926, '12': 1627, '00': 56}
Ireland+UK %: 14.375351594616816


168953